# CountYOLO — Fine-tune nhánh thuần YOLO-World (Text-only CountGD-Box Pipeline) trên Colab A100

Notebook này huấn luyện nhánh **thuần YOLO-World (`YOLOWorldCounting`)** theo chuẩn pipeline CountGD-Box:
1. **Multi-class prompt contrastive (`--max-labels 80`)**: mỗi ảnh ghép 1 prompt dương + tối đa 79 prompt âm từ tập train (theo `odvg.py`), ngăn hiện tượng đếm khống khi vật thể không có trong ảnh.
2. **Bộ prompt chuẩn `FSC-147-D`**: danh từ số ít và sửa lỗi ngữ nghĩa (`polka dot`, `package of fresh cut fruit`, ...).
3. **Data Augmentation chuẩn CountVid**: `RandomHorizontalFlip` + 50% `RandomResize([400,500,600]) -> RandomSizeCrop(384,600) -> RandomResize(480..800)`.
4. **Chuẩn hóa gradient theo tổng GT của logical batch 4 ảnh** (`SetCriterion`).

> **Hướng dẫn:** Trên thanh menu Colab chọn **Runtime → Change runtime type → A100 GPU**, sau đó bấm **Runtime → Run all**.

In [ ]:
# 1. Kiểm tra GPU A100 và kết nối Google Drive để lưu kết quả lâu dài
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), 'Hãy bật GPU (A100) trong Runtime -> Change runtime type!'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('GPU:', torch.cuda.get_device_name(0))

MOUNT_DRIVE = True  # Đặt False nếu không muốn lưu sang Google Drive
OUT_ROOT = '/content/outputs/yoloworld_counting_v2'
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/countyolo_runs/yoloworld_counting_v2'
print('Thư mục lưu checkpoint & kết quả:', OUT_ROOT)

In [ ]:
# 2. Cài đặt thư viện & clone nhánh experiment/yoloworld-countgd-adapt
!pip install -q setuptools==80.9.0 wheel==0.45.1
!pip install -q ultralytics==8.4.177 "numpy<2.3" scipy opencv-python kagglehub
!pip install -q --no-build-isolation git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33

!rm -rf /content/countyolo
!git clone -b experiment/yoloworld-countgd-adapt https://github.com/Thien-Dan/countyolo.git /content/countyolo
%cd /content/countyolo

In [ ]:
# 3. Tải dataset FSC-147 (qua kagglehub public hoặc từ Google Drive) và weights yolov8s-worldv2.pt
import glob, pathlib, urllib.request
import kagglehub

DRIVE_FSC147 = '/content/drive/MyDrive/FSC147'
if pathlib.Path(DRIVE_FSC147, 'Train_Test_Val_FSC_147.json').exists():
    DATA_ROOT = pathlib.Path(DRIVE_FSC147)
else:
    dl_path = kagglehub.dataset_download('xuncngng/fsc147-0')
    matches = list(pathlib.Path(dl_path).rglob('Train_Test_Val_FSC_147.json'))
    assert len(matches) == 1, f'Không tìm thấy FSC147 trong {dl_path}'
    DATA_ROOT = matches[0].parent
print('DATA_ROOT:', DATA_ROOT)

WEIGHTS = pathlib.Path('/content/yolov8s-worldv2.pt')
if not WEIGHTS.exists():
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8s-worldv2.pt',
        WEIGHTS
    )
print('WEIGHTS:', WEIGHTS, WEIGHTS.stat().st_size, 'bytes')

In [ ]:
# 4. Chạy Unit Tests & Gradient Smoke Test trước khi train
!python -m unittest tools.test_yoloworld_counting tools.test_yoloworld_resume
!rm -rf /content/smoke_check
!python -u tools/train_yoloworld_counting.py \
    --data-root "{DATA_ROOT}" \
    --weights "{WEIGHTS}" \
    --output /content/smoke_check \
    --smoke

In [ ]:
# 5. Huấn luyện 30 Epoch trên A100 (tự động lưu best.pth, last.pth, history.json và đánh giá full-val + absent prompt)
!python -u tools/train_yoloworld_counting.py \
    --data-root "{DATA_ROOT}" \
    --weights "{WEIGHTS}" \
    --output "{OUT_ROOT}" \
    --epochs 30 \
    --accumulate 4 \
    --neg-weight 0.15 \
    --neg-topk 32 \
    --neighbor-sigma 0.004 \
    --neck-lr 2e-5 \
    --stride-radius 4.0

In [ ]:
# 6. Xem kết quả tổng kết sau 30 epoch
import json, pathlib
out = pathlib.Path(OUT_ROOT)
print('=== RESULT ===')
print((out / 'result.json').read_text())
print('=== ABSENT PROMPT DIAGNOSTIC (elephants on non-elephant images) ===')
print((out / 'adapted_absent.json').read_text())